# Endpoint Alert Cleaning and Explainable Risk Analysis

**Scope:** clean `track2_endpoint_alerts.xlsx`, validate endpoint telemetry, enrich it with the identity/asset reference, and identify risky devices/users. This notebook is intentionally focused on Issue #6; dashboard development, full IAM/firewall integration, and an AI agent are out of scope here.

**Reproducibility:** run all cells from top to bottom. Raw columns are preserved with `_raw` suffixes, cleaning is deterministic, every quality concern is flagged rather than silently discarded, and processed outputs are written to `data/processed/` and `reports/`.


## 1. Imports, paths, and constants

The notebook expects both input files in the repository `data/raw/` directory. This keeps execution portable for teammates and reviewers.

### Imports


In [4]:
from pathlib import Path
import re
import json
import hashlib
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

### Paths and constants


In [ ]:
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINT_NAME = "track2_endpoint_alerts.xlsx"
IDENTITY_NAME = "track2_identity_asset_master.csv"

### Input resolver


In [6]:
def resolve_input(name: str) -> Path:
    candidate = RAW_DIR / name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(
        f"Could not find {name}. Place it in {RAW_DIR}."
    )

ENDPOINT_PATH = resolve_input(ENDPOINT_NAME)
IDENTITY_PATH = resolve_input(IDENTITY_NAME)
print("Endpoint:", ENDPOINT_PATH)
print("Identity reference:", IDENTITY_PATH)

EXPECTED_ENDPOINT_COLUMNS = {
    "alert_id", "detected_timestamp", "resolved_timestamp", "hostname", "user_id",
    "endpoint_product", "alert_name", "severity", "status", "description",
    "file_path", "process_name", "sha256", "assigned_to", "device_criticality"
}
ALLOWED_SEVERITY = {"Critical", "High", "Medium", "Low"}
ALLOWED_STATUS = {"New", "Open", "In Progress", "Resolved", "Closed", "Unassigned"}
ALLOWED_CRITICALITY = {"Critical", "High", "Medium", "Low"}
MISSING_TOKENS = {"", "NA", "N/A", "NONE", "NULL", "NAN", "NOT AVAILABLE", "UNKNOWN"}

Endpoint: c:\Users\aadhi\.copilot\chats\2026-09-13\attached-file-track2-dataset-notes-txt-c-f0ad7025\data\raw\track2_endpoint_alerts.xlsx
Identity reference: c:\Users\aadhi\.copilot\chats\2026-09-13\attached-file-track2-dataset-notes-txt-c-f0ad7025\data\raw\track2_identity_asset_master.csv


## 2. Load and profile the raw endpoint data

This establishes the baseline before any transformation. Duplicate alert IDs are retained and explicitly flagged because repeated IDs can be a data-quality issue or a meaningful repeated observation.

### Load raw data


In [45]:
endpoint_raw = pd.read_excel(ENDPOINT_PATH)
endpoint_raw.head()

,alert_id,detected_timestamp,resolved_timestamp,hostname,user_id,endpoint_product,alert_name,severity,status,description,file_path,process_name,sha256,assigned_to,device_criticality
0,EPA00001920,2026-08-09T11:06:47,2026-08-11T06:06:47,VDR-11768,EMP11768,Kaspersky,Lateral movement detected,Medium,R,Endpoint protection detected suspicious activi...,C:\ProgramData\svc\agent.dll,NaN,054e0398bbd02a8260a7d6d34e2f84e7ee67706f95b453...,NaN,NaN
1,EPA00004656,08/09/2026 17:32,NaN,VDR-11100,EMP-11100,Quick Heal,MALWARE_DETECTED,HIGH,WIP,Multiple failed attempts to disable security a...,NaN,msedge.exe,20e2f44e68c92f4a824a1a67bb3b9316e6843372c31e02...,soc.analyst2,CRITICAL
2,EPA00001840,08/09/2026,13-Sep-2026 13:51:20,VDR-10029,EMP10029,McAfee,Unusual admin share access,Severe,IN_PROGRESS,Malicious file blocked. User notified. Need va...,E:\Backup\data.rar,explorer.exe,abbc24bffe28684c1d48bbe075610913ff0accfbc1efc9...,unassigned,NaN
3,EPA00004878,1786065442,11/08/2026 14:17,WS-10202.corp.local,EMP-10202,Windows Defender,MALWARE_DETECTED,MEDIUM,Unassigned,Alert generated from heuristic engine. Possibl...,C:\ProgramData\svc\agent.dll,winword.exe,99be2fd35ebc37d1e43fafbbf706101154ca19ecff6c12...,incident.team,CRITICAL
4,EPA00003821,2026-09-09 09:26:10,NaN,lpt-10538,EMP10538,Kaspersky,Ransomware Activity,M,IN_PROGRESS,Multiple failed attempts to disable security a...,E:\Backup\data.rar,winword.exe,NaN,admin,NaN


### Validate schema


In [46]:
missing_columns = EXPECTED_ENDPOINT_COLUMNS - set(endpoint_raw.columns)
if missing_columns:
    raise ValueError(f"Missing required endpoint columns: {sorted(missing_columns)}")

### Raw baseline


In [9]:
endpoint_raw = endpoint_raw.copy()
endpoint_raw["_source_row_number"] = np.arange(2, len(endpoint_raw) + 2)

### Profile missing values


In [10]:
profile_rows = []
for column in endpoint_raw.columns:
    values = endpoint_raw[column]
    profile_rows.append({
        "column": column,
        "dtype": str(values.dtype),
        "row_count": len(values),
        "missing_count": int(values.isna().sum()),
        "missing_percent": round(float(values.isna().mean() * 100), 2),
        "unique_count": int(values.nunique(dropna=True)),
    })
raw_profile = pd.DataFrame(profile_rows)

### Profile categorical fields


In [11]:
exact_duplicate_count = int(endpoint_raw.duplicated(subset=list(EXPECTED_ENDPOINT_COLUMNS)).sum())
duplicate_alert_id_count = int(endpoint_raw["alert_id"].duplicated(keep=False).sum())
raw_baseline = pd.DataFrame([{
    "raw_rows": len(endpoint_raw),
    "unique_alert_ids": endpoint_raw["alert_id"].nunique(),
    "exact_duplicate_rows": exact_duplicate_count,
    "rows_with_duplicate_alert_id": duplicate_alert_id_count,
}])

display(raw_baseline)
display(raw_profile.sort_values(["missing_count", "column"], ascending=[False, True]))

,raw_rows,unique_alert_ids,exact_duplicate_rows,rows_with_duplicate_alert_id
0,8240,8000,240,480


,column,dtype,row_count,missing_count,missing_percent,unique_count
13,assigned_to,str,8240,3138,38.08,5
2,resolved_timestamp,str,8240,2405,29.19,4716
10,file_path,str,8240,1675,20.33,8
11,process_name,str,8240,1495,18.14,9
12,sha256,str,8240,1484,18.01,6562
14,device_criticality,str,8240,1299,15.76,11
1,detected_timestamp,str,8240,1006,12.21,5811
3,hostname,str,8240,503,6.10,4345
15,_source_row_number,int64,8240,0,0.00,8240
0,alert_id,str,8240,0,0.00,8000


### Step


In [12]:
for column in ["severity", "status", "device_criticality", "alert_name", "endpoint_product"]:
    print(f"\\n{column} values")
    display(endpoint_raw[column].value_counts(dropna=False).rename_axis(column).reset_index(name="count").head(30))

\nseverity values


,severity,count
0,MEDIUM,679
1,P3,640
2,Medium,627
3,M,607
4,Moderate,599
5,LOW,525
6,Minor,506
7,P4,501
8,L,484
9,Low,477


\nstatus values


,status,count
0,New,561
1,N,511
2,Resolved,487
3,Open,478
4,NEW,477
5,Unassigned,473
6,O,467
7,OPEN,446
8,Active,442
9,Closed,415


\ndevice_criticality values


,device_criticality,count
0,NaN,1299
1,Critical,677
2,Low,662
3,L,661
4,High,658
5,LOW,636
6,CRITICAL,617
7,H,617
8,Medium,616
9,M,611


\nalert_name values


,alert_name,count
0,MALWARE_DETECTED,588
1,Trojan:Win32,579
2,Browser hijack attempt,570
3,USB device blocked,562
4,ransomware_behavior,560
5,Unusual admin share access,558
6,Suspicious PowerShell Execution,549
7,Malware detected,547
8,suspicious_powershell,542
9,Unauthorized USB,538


\nendpoint_product values


,endpoint_product,count
0,SYMANTEC,963
1,SentinelOne,939
2,Defender,934
3,Windows Defender,916
4,McAfee,907
5,Quick Heal,904
6,Kaspersky,896
7,Carbon Black,891
8,CrowdStrike,890


## 3. Cleaning helper functions

Raw values remain available for audit. The functions deliberately return `pd.NA` for recognized missing tokens and do not manufacture identifiers or timestamps.

### Missing values


In [13]:
def clean_missing(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    return pd.NA if text.upper() in MISSING_TOKENS else text


def clean_user_id(value):
    value = clean_missing(value)
    if pd.isna(value):
        return pd.NA
    compact = re.sub(r"[^A-Z0-9]", "", str(value).upper())
    if compact.isdigit():
        compact = "EMP" + compact
    return compact or pd.NA


def clean_hostname(value):
    value = clean_missing(value)
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    value = re.sub(r"\\.CORP\\.LOCAL$", "", value)
    return value.replace("_", "-") or pd.NA

### Identifiers


In [14]:
def clean_label(value):
    value = clean_missing(value)
    return pd.NA if pd.isna(value) else re.sub(r"\\s+", " ", str(value)).strip()

### Labels


In [15]:
def canonical_map(value, mapping):
    value = clean_missing(value)
    if pd.isna(value):
        return pd.NA
    key = str(value).strip().upper()
    return mapping.get(key, pd.NA)

### Mappings


In [16]:
SEVERITY_MAP = {
    "P1": "Critical", "S1": "Critical", "CRIT": "Critical", "CRITICAL": "Critical", "SEVERE": "Critical",
    "P2": "High", "S2": "High", "H": "High", "HIGH": "High", "MAJOR": "High",
    "P3": "Medium", "S3": "Medium", "MEDIUM": "Medium", "MODERATE": "Medium", "M": "Medium",
    "P4": "Low", "S4": "Low", "LOW": "Low", "MINOR": "Low", "L": "Low",
}
STATUS_MAP = {
    "NEW": "New", "N": "New", "OPEN": "Open", "O": "Open", "ACTIVE": "Open",
    "IN_PROGRESS": "In Progress", "IN PROGRESS": "In Progress", "WIP": "In Progress", "INVESTIGATING": "In Progress",
    "RESOLVED": "Resolved", "R": "Resolved", "CLOSED": "Closed", "C": "Closed", "NOT MALICIOUS": "Closed", "FALSE_POSITIVE": "Closed", "FALSE POSITIVE": "Closed", "FP": "Closed",
    "UNASSIGNED": "Unassigned", "U": "Unassigned",
}
CRITICALITY_MAP = {
    "CRITICAL": "Critical", "C": "Critical", "P1": "Critical",
    "HIGH": "High", "H": "High", "P2": "High",
    "MEDIUM": "Medium", "M": "Medium", "P3": "Medium",
    "LOW": "Low", "L": "Low", "P4": "Low",
}

In [17]:
# Displayable mapping tables make normalization auditable.
def mapping_table(mapping, field_name):
    return pd.DataFrame([{"field": field_name, "raw_value": raw, "clean_value": clean} for raw, clean in mapping.items()])

mapping_tables = pd.concat([
    mapping_table(SEVERITY_MAP, "severity"),
    mapping_table(STATUS_MAP, "status"),
    mapping_table(CRITICALITY_MAP, "device_criticality"),
], ignore_index=True)
display(mapping_tables)

ALERT_CATEGORY_RULES = [
    ("Malware/Ransomware", r"malware|ransomware|trojan"),
    ("Credential Access", r"credential|keylogger"),
    ("Lateral Movement", r"lateral|admin share"),
    ("PowerShell", r"powershell"),
    ("USB", r"usb"),
    ("Security Tampering", r"disable security|tamper"),
    ("Phishing/Browser Threat", r"phishing|browser hijack"),
]

def classify_alert_category(alert_name):
    text = str(alert_name or "").lower()
    for category, pattern in ALERT_CATEGORY_RULES:
        if re.search(pattern, text):
            return category
    return "Other"


,field,raw_value,clean_value
0,severity,P1,Critical
1,severity,S1,Critical
2,severity,CRIT,Critical
3,severity,CRITICAL,Critical
4,severity,SEVERE,Critical
5,severity,P2,High
6,severity,S2,High
7,severity,H,High
8,severity,HIGH,High
9,severity,MAJOR,High


### Timestamp and hash validators


In [18]:
def parse_timestamp(value):
    value = clean_missing(value)
    if pd.isna(value):
        return pd.NaT
    text = str(value).strip()
    if re.fullmatch(r"\\d{10}(?:\\.0)?", text):
        try:
            return pd.to_datetime(int(float(text)), unit="s", errors="coerce")
        except (TypeError, ValueError, OverflowError):
            return pd.NaT
    formats = [
        "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d",
        "%Y/%m/%d %H:%M:%S", "%Y/%m/%d %H:%M", "%Y/%m/%d",
        "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%Y",
        "%d-%b-%Y %H:%M:%S", "%d-%b-%Y %H:%M", "%d-%b-%Y",
        "%m/%d/%Y %H:%M:%S", "%m/%d/%Y %H:%M", "%m/%d/%Y",
        "%m-%d-%Y %I:%M:%S %p", "%m-%d-%Y %I:%M %p",
    ]
    for fmt in formats:
        try:
            return pd.Timestamp(datetime.strptime(text, fmt))
        except ValueError:
            continue
    return pd.NaT

### Step


In [19]:
def valid_sha256(value):
    value = clean_missing(value)
    if pd.isna(value):
        return pd.NA
    return bool(re.fullmatch(r"[0-9A-Fa-f]{64}", str(value).strip()))

## 4. Clean endpoint fields and create quality flags

### Preserve raw columns


In [20]:
endpoint = endpoint_raw.copy()
for column in EXPECTED_ENDPOINT_COLUMNS:
    endpoint[f"{column}_raw"] = endpoint[column]

### Clean identifiers and categories


In [21]:
endpoint["user_id_clean"] = endpoint["user_id"].map(clean_user_id)
endpoint["hostname_clean"] = endpoint["hostname"].map(clean_hostname)
endpoint["severity_clean"] = endpoint["severity"].map(lambda x: canonical_map(x, SEVERITY_MAP))
endpoint["status_clean"] = endpoint["status"].map(lambda x: canonical_map(x, STATUS_MAP))
endpoint["device_criticality_clean"] = endpoint["device_criticality"].map(lambda x: canonical_map(x, CRITICALITY_MAP))
for column in ["endpoint_product", "alert_name", "description", "file_path", "process_name", "assigned_to"]:
    endpoint[f"{column}_clean"] = endpoint[column].map(clean_label)

### Parse timestamps and hashes


In [22]:
endpoint["detected_timestamp_clean"] = endpoint["detected_timestamp"].map(parse_timestamp)
endpoint["resolved_timestamp_clean"] = endpoint["resolved_timestamp"].map(parse_timestamp)
endpoint["sha256_valid_flag"] = endpoint["sha256"].map(valid_sha256).fillna(False).astype(bool)
endpoint["sha256_missing_flag"] = endpoint["sha256"].map(clean_missing).isna()

### Quality flags


In [23]:
endpoint["duplicate_alert_id_flag"] = endpoint["alert_id"].duplicated(keep=False)
endpoint["exact_duplicate_flag"] = endpoint.duplicated(subset=list(EXPECTED_ENDPOINT_COLUMNS), keep=False)
endpoint["duplicate_alert_type"] = np.select(
    [endpoint["exact_duplicate_flag"], endpoint["duplicate_alert_id_flag"]],
    ["duplicate_exact_record", "duplicate_alert_id_different_content"],
    default="unique_alert_id"
)
endpoint["missing_user_id_flag"] = endpoint["user_id_clean"].isna()
endpoint["missing_hostname_flag"] = endpoint["hostname_clean"].isna()
endpoint["missing_detected_timestamp_flag"] = endpoint["detected_timestamp_clean"].isna()
endpoint["invalid_detected_timestamp_flag"] = endpoint["detected_timestamp"].map(clean_missing).notna() & endpoint["detected_timestamp_clean"].isna()
endpoint["missing_resolved_timestamp_flag"] = endpoint["resolved_timestamp_clean"].isna()
endpoint["invalid_resolved_timestamp_flag"] = endpoint["resolved_timestamp"].map(clean_missing).notna() & endpoint["resolved_timestamp_clean"].isna()
endpoint["impossible_resolution_flag"] = (
    endpoint["detected_timestamp_clean"].notna()
    & endpoint["resolved_timestamp_clean"].notna()
    & (endpoint["resolved_timestamp_clean"] < endpoint["detected_timestamp_clean"])
)
endpoint["unresolved_alert_flag"] = endpoint["status_clean"].isin(["New", "Open", "In Progress", "Unassigned"])
endpoint["resolved_without_timestamp_flag"] = endpoint["status_clean"].isin(["Resolved", "Closed"]) & endpoint["resolved_timestamp_clean"].isna()
endpoint["unresolved_with_resolution_timestamp_flag"] = endpoint["unresolved_alert_flag"] & endpoint["resolved_timestamp_clean"].notna()

### Mapping flags


In [24]:
endpoint["unmapped_severity_flag"] = endpoint["severity"].map(clean_missing).notna() & endpoint["severity_clean"].isna()
endpoint["unmapped_status_flag"] = endpoint["status"].map(clean_missing).notna() & endpoint["status_clean"].isna()
endpoint["unmapped_criticality_flag"] = endpoint["device_criticality"].map(clean_missing).notna() & endpoint["device_criticality_clean"].isna()
endpoint["invalid_sha256_flag"] = ~endpoint["sha256_valid_flag"] & ~endpoint["sha256_missing_flag"]
endpoint["sha256_validation_status"] = np.select(
    [endpoint["sha256_missing_flag"], endpoint["sha256_valid_flag"]],
    ["missing_sha256", "valid_sha256"],
    default="malformed_sha256"
)

### Security categories


In [25]:
endpoint["alert_name_lower"] = endpoint["alert_name_clean"].astype("string").str.lower()
endpoint["alert_category"] = endpoint["alert_name_clean"].map(classify_alert_category)
endpoint["malware_or_ransomware_flag"] = endpoint["alert_category"].eq("Malware/Ransomware")
endpoint["powershell_flag"] = endpoint["alert_category"].eq("PowerShell")
endpoint["lateral_movement_flag"] = endpoint["alert_category"].eq("Lateral Movement")
endpoint["usb_flag"] = endpoint["alert_category"].eq("USB")
endpoint["tampering_flag"] = endpoint["alert_category"].eq("Security Tampering")
display(endpoint["alert_category"].value_counts().rename_axis("alert_category").reset_index(name="alerts"))

,alert_category,alerts
0,Malware/Ransomware,2811
1,USB,1100
2,Lateral Movement,1094
3,PowerShell,1091
4,Phishing/Browser Threat,1086
5,Credential Access,1058


### Quality status


In [26]:
quality_flags = [
    "duplicate_alert_id_flag", "exact_duplicate_flag", "missing_user_id_flag", "missing_hostname_flag",
    "invalid_detected_timestamp_flag", "invalid_resolved_timestamp_flag", "impossible_resolution_flag",
    "resolved_without_timestamp_flag", "unresolved_with_resolution_timestamp_flag",
    "unmapped_severity_flag", "unmapped_status_flag", "unmapped_criticality_flag",
    "sha256_missing_flag", "invalid_sha256_flag"
]
endpoint["data_quality_issue_count"] = endpoint[quality_flags].astype(int).sum(axis=1)
endpoint["data_quality_status"] = np.where(endpoint["data_quality_issue_count"].eq(0), "Pass", "Review")
print("Cleaned endpoint shape:", endpoint.shape)
display(endpoint[["alert_id", "user_id_raw", "user_id_clean", "hostname_raw", "hostname_clean", "severity_clean", "status_clean", "alert_category", "data_quality_status"]].head())

Cleaned endpoint shape: (8240, 73)


,alert_id,user_id_raw,user_id_clean,hostname_raw,hostname_clean,severity_clean,status_clean,alert_category,data_quality_status
0,EPA00001920,EMP11768,EMP11768,VDR-11768,VDR-11768,Medium,Resolved,Lateral Movement,Pass
1,EPA00004656,EMP-11100,EMP11100,VDR-11100,VDR-11100,High,In Progress,Malware/Ransomware,Pass
2,EPA00001840,EMP10029,EMP10029,VDR-10029,VDR-10029,Critical,In Progress,Lateral Movement,Review
3,EPA00004878,EMP-10202,EMP10202,WS-10202.corp.local,WS-10202.CORP.LOCAL,Medium,Unassigned,Malware/Ransomware,Review
4,EPA00003821,EMP10538,EMP10538,lpt-10538,LPT-10538,Medium,In Progress,Malware/Ransomware,Review


## 5. Data-quality report and validation

This report is an acceptance artifact for Issue #6. It separates missingness from invalidity and preserves row-level evidence.

### Quality metrics


In [27]:
quality_metrics = [
    {"metric": "raw_rows", "value": len(endpoint_raw)},
    {"metric": "cleaned_rows", "value": len(endpoint)},
    {"metric": "unique_alert_ids", "value": endpoint["alert_id"].nunique()},
    {"metric": "rows_with_duplicate_alert_id", "value": int(endpoint["duplicate_alert_id_flag"].sum())},
    {"metric": "duplicate_alert_id_excess_rows", "value": int(len(endpoint) - endpoint["alert_id"].nunique())},
    {"metric": "exact_duplicate_rows", "value": int(endpoint["exact_duplicate_flag"].sum())},
    {"metric": "duplicate_alert_id_different_content", "value": int((endpoint["duplicate_alert_type"] == "duplicate_alert_id_different_content").sum())},
    {"metric": "missing_user_id", "value": int(endpoint["missing_user_id_flag"].sum())},
    {"metric": "missing_hostname", "value": int(endpoint["missing_hostname_flag"].sum())},
    {"metric": "invalid_detected_timestamp", "value": int(endpoint["invalid_detected_timestamp_flag"].sum())},
    {"metric": "invalid_resolved_timestamp", "value": int(endpoint["invalid_resolved_timestamp_flag"].sum())},
    {"metric": "impossible_resolution", "value": int(endpoint["impossible_resolution_flag"].sum())},
    {"metric": "resolved_without_timestamp", "value": int(endpoint["resolved_without_timestamp_flag"].sum())},
    {"metric": "unresolved_with_resolution_timestamp", "value": int(endpoint["unresolved_with_resolution_timestamp_flag"].sum())},
    {"metric": "invalid_sha256", "value": int((~endpoint["sha256_valid_flag"] & ~endpoint["sha256_missing_flag"]).sum())},
    {"metric": "unmapped_severity", "value": int(endpoint["unmapped_severity_flag"].sum())},
    {"metric": "unmapped_status", "value": int(endpoint["unmapped_status_flag"].sum())},
    {"metric": "unmapped_criticality", "value": int(endpoint["unmapped_criticality_flag"].sum())},
    {"metric": "rows_requiring_review", "value": int(endpoint["data_quality_status"].eq("Review").sum())},
]

### Quality report


In [28]:
quality_report = pd.DataFrame(quality_metrics)
display(quality_report)

# Hard checks: unexpected canonical values are implementation errors, not data to hide.
assert set(endpoint["severity_clean"].dropna().unique()).issubset(ALLOWED_SEVERITY)
assert set(endpoint["status_clean"].dropna().unique()).issubset(ALLOWED_STATUS)
assert set(endpoint["device_criticality_clean"].dropna().unique()).issubset(ALLOWED_CRITICALITY)
assert not (endpoint["impossible_resolution_flag"] & ~(endpoint["detected_timestamp_clean"].notna() & endpoint["resolved_timestamp_clean"].notna())).any()

,metric,value
0,raw_rows,8240
1,cleaned_rows,8240
2,unique_alert_ids,8000
3,rows_with_duplicate_alert_id,480
4,duplicate_alert_id_excess_rows,240
5,exact_duplicate_rows,480
6,duplicate_alert_id_different_content,0
7,missing_user_id,0
8,missing_hostname,503
9,invalid_detected_timestamp,819


### Quality breakdown

In [29]:
display(endpoint["duplicate_alert_type"].value_counts().rename_axis("duplicate_alert_type").reset_index(name="rows"))
display(endpoint["sha256_validation_status"].value_counts().rename_axis("sha256_validation_status").reset_index(name="rows"))
display(endpoint[["status_clean", "resolved_without_timestamp_flag", "unresolved_with_resolution_timestamp_flag"]].value_counts().reset_index(name="rows"))

,duplicate_alert_type,rows
0,unique_alert_id,7760
1,duplicate_exact_record,480


,sha256_validation_status,rows
0,valid_sha256,5941
1,missing_sha256,1484
2,malformed_sha256,815


,status_clean,resolved_without_timestamp_flag,unresolved_with_resolution_timestamp_flag,rows
0,Open,False,True,1156
1,Closed,False,False,1019
2,New,False,True,985
3,In Progress,False,True,935
4,Resolved,False,False,794
5,Open,False,False,677
6,Closed,True,False,609
7,New,False,False,564
8,In Progress,False,False,548
9,Resolved,True,False,480


## 6. Identity reference cleaning and endpoint enrichment

This notebook uses the teammate's identity-master work as a reference, without modifying their notebook. The same join-key normalization is applied locally so endpoint enrichment is reproducible. Existing identity conflicts are preserved and flagged.

### Load identity data


In [30]:
identity = pd.read_csv(IDENTITY_PATH)
required_identity = {"user_id", "hostname"}
missing_identity = required_identity - set(identity.columns)
if missing_identity:
    raise ValueError(f"Missing required identity columns: {sorted(missing_identity)}")

### Normalize identity keys


In [31]:
identity = identity.copy()
identity["user_id_clean"] = identity["user_id"].map(clean_user_id)
identity["hostname_clean"] = identity["hostname"].map(clean_hostname)

### Device attributes


In [32]:
if "device_id" in identity.columns:
    identity["device_id_clean"] = identity["device_id"].map(clean_label).astype("string").str.upper().str.replace(r"[\\s-]+", "", regex=True)

### Employee status


In [33]:
if "status" in identity.columns:
    identity_status_map = {
        "A": "Active", "ACTIVE": "Active", "ENABLED": "Active", "LIVE": "Active", "WORKING": "Active",
        "D": "Disabled", "DISABLED": "Disabled", "BLOCKED": "Disabled", "DEACTIVATED": "Disabled",
        "L": "Inactive", "LEFT": "Inactive", "RESIGNED": "Inactive", "TERMINATED": "Inactive", "EXITED": "Inactive",
        "ON_LEAVE": "On Leave", "ON LEAVE": "On Leave", "OOO": "On Leave",
    }
    identity["status_clean"] = identity["status"].map(lambda x: canonical_map(x, identity_status_map))

### Departments


In [34]:
if "department" in identity.columns:
    identity["department_clean"] = identity["department"].map(clean_label)
    identity["department_clean"] = identity["department_clean"].astype("string").str.strip().str.replace(r"\\s+", " ", regex=True).str.title()

### Ambiguous-key checks


In [35]:
user_counts = identity.groupby("user_id_clean", dropna=True)["user_id_clean"].transform("size")
host_counts = identity.groupby("hostname_clean", dropna=True)["hostname_clean"].transform("size")
identity["ambiguous_user_key_flag"] = user_counts.gt(1).fillna(False)
identity["ambiguous_hostname_key_flag"] = host_counts.gt(1).fillna(False)
ambiguous_user_keys = set(identity.loc[identity["ambiguous_user_key_flag"], "user_id_clean"].dropna())
ambiguous_hostname_keys = set(identity.loc[identity["ambiguous_hostname_key_flag"], "hostname_clean"].dropna())
print("Ambiguous user keys:", len(ambiguous_user_keys))
print("Ambiguous hostname keys:", len(ambiguous_hostname_keys))

Ambiguous user keys: 90
Ambiguous hostname keys: 87


### Join preparation


In [36]:
identity_user = identity.dropna(subset=["user_id_clean"]).drop_duplicates("user_id_clean", keep=False).copy()
identity_host = identity.dropna(subset=["hostname_clean"]).drop_duplicates("hostname_clean", keep=False).copy()
user_attributes = [c for c in ["full_name", "username", "department_clean", "role", "location", "status_clean", "device_id_clean", "manager_username"] if c in identity]
host_attributes = [c for c in ["device_id_clean", "status_clean", "department_clean"] if c in identity]

user_join = identity_user[["user_id_clean"] + user_attributes].rename(columns={c: f"user_{c}" for c in user_attributes})

### Identity enrichment


In [37]:
host_join = identity_host[["hostname_clean"] + host_attributes].rename(columns={c: f"host_{c}" for c in host_attributes})


### Join summary


In [38]:
# Merge only against unambiguous identity keys to prevent row multiplication.
enriched = endpoint.merge(user_join, on="user_id_clean", how="left", validate="many_to_one")
enriched = enriched.merge(host_join, on="hostname_clean", how="left", validate="many_to_one")
enriched["user_key_match_flag"] = enriched[[c for c in enriched.columns if c.startswith("user_")]].notna().any(axis=1)
enriched["hostname_key_match_flag"] = enriched[[c for c in enriched.columns if c.startswith("host_")]].notna().any(axis=1)
enriched["ambiguous_user_key_flag"] = enriched["user_id_clean"].isin(ambiguous_user_keys)
enriched["ambiguous_hostname_key_flag"] = enriched["hostname_clean"].isin(ambiguous_hostname_keys)
enriched["join_match_type"] = np.select(
    [
        enriched["ambiguous_user_key_flag"] | enriched["ambiguous_hostname_key_flag"],
        enriched["user_key_match_flag"] & enriched["hostname_key_match_flag"],
        enriched["user_key_match_flag"],
        enriched["hostname_key_match_flag"],
    ],
    ["ambiguous_match", "both_keys_match", "user_only_match", "hostname_only_match"],
    default="unmatched"
)
enriched["identity_join_review_flag"] = enriched["join_match_type"].isin(["unmatched", "user_only_match", "hostname_only_match", "ambiguous_match"])
enriched["inactive_user_activity_flag"] = enriched.get("user_status_clean", pd.Series(pd.NA, index=enriched.index)).isin(["Inactive", "Disabled"])


### Step


In [39]:
enrichment_summary = enriched["join_match_type"].value_counts(dropna=False).rename_axis("join_match_type").reset_index(name="rows")
display(enrichment_summary)

,join_match_type,rows
0,both_keys_match,5932
1,user_only_match,2047
2,ambiguous_match,261


## 7. Explainable risky-device and risky-user analysis

The score is intentionally transparent and is a triage aid, not a claim of compromise. Components are retained so an analyst can explain every ranking.

### Risk helpers


In [40]:
def bool_sum(series):
    return int(series.fillna(False).astype(bool).sum())

### Risk aggregation


In [41]:
def aggregate_risk(frame, key_column, display_column):
    grouped = []
    for key, group in frame.groupby(key_column, dropna=False):
        if pd.isna(key):
            continue
        critical = int((group["severity_clean"] == "Critical").sum())
        high = int((group["severity_clean"] == "High").sum())
        unresolved_critical = int((group["severity_clean"].eq("Critical") & group["unresolved_alert_flag"]).sum())
        unresolved = int(group["unresolved_alert_flag"].sum())
        malware = bool_sum(group["malware_or_ransomware_flag"])
        powershell = bool_sum(group["powershell_flag"])
        lateral = bool_sum(group["lateral_movement_flag"])
        usb = bool_sum(group["usb_flag"])
        tampering = bool_sum(group["tampering_flag"])
        credential = int(group["alert_category"].eq("Credential Access").sum())
        impossible = bool_sum(group["impossible_resolution_flag"])
        inactive = bool_sum(group["inactive_user_activity_flag"])
        identity_review = bool_sum(group["identity_join_review_flag"])
        score_components = {
            "critical_severity_points": 5 * critical,
            "high_severity_points": 3 * high,
            "unresolved_critical_points": 3 * unresolved_critical,
            "malware_ransomware_points": 3 * malware,
            "powershell_points": 2 * powershell,
            "lateral_movement_points": 2 * lateral,
            "credential_access_points": 2 * credential,
            "tampering_points": 2 * tampering,
            "usb_points": usb,
            "impossible_resolution_points": impossible,
            "inactive_user_points": 3 * inactive,
            "identity_review_points": identity_review,
        }
        score = sum(score_components.values())
        reasons = []
        for label, count in [("critical alert(s)", critical), ("unresolved critical alert(s)", unresolved_critical), ("malware/ransomware alert(s)", malware), ("PowerShell alert(s)", powershell), ("lateral-movement alert(s)", lateral), ("credential-access alert(s)", credential), ("inactive-user event(s)", inactive), ("identity review concern(s)", identity_review)]:
            if count: reasons.append(f"{count} {label}")
        row = {display_column: key, "alert_count": len(group), "critical_alert_count": critical, "high_alert_count": high, "unresolved_alert_count": unresolved, "unresolved_critical_count": unresolved_critical, "malware_or_ransomware_count": malware, "powershell_count": powershell, "lateral_movement_count": lateral, "credential_access_count": credential, "tampering_count": tampering, "usb_count": usb, "impossible_resolution_count": impossible, "inactive_user_activity_count": inactive, "identity_join_review_count": identity_review, "associated_user_count": group["user_id_clean"].nunique(dropna=True), "risk_score": score, "risk_reasons": "; ".join(reasons) if reasons else "volume/low-severity activity only"}
        row.update(score_components)
        grouped.append(row)
    result = pd.DataFrame(grouped)
    if result.empty:
        return result
    result["risk_band"] = pd.cut(result["risk_score"], bins=[-np.inf, 4, 9, 19, np.inf], labels=["Low", "Medium", "High", "Critical"], right=True).astype(str)
    return result.sort_values(["risk_score", "alert_count"], ascending=False).reset_index(drop=True)

### Risk tables and reference fields


In [42]:
enriched["device_key"] = enriched["hostname_clean"].fillna("MISSING_HOST::" + enriched["user_id_clean"].fillna("UNKNOWN").astype(str))
risky_devices = aggregate_risk(enriched, "device_key", "device_key")
risky_users = aggregate_risk(enriched, "user_id_clean", "user_id_clean")

# Add useful reference fields to the ranked outputs.
device_reference = enriched.groupby("device_key", dropna=False).agg(
    hostname=("hostname_clean", "first"),
    device_criticality=("device_criticality_clean", "first"),
    departments=("user_department_clean", lambda s: ", ".join(sorted(set(str(x) for x in s.dropna())))),
).reset_index()
risky_devices = risky_devices.merge(device_reference, on="device_key", how="left")
user_reference = enriched.groupby("user_id_clean", dropna=False).agg(
    full_name=("user_full_name", "first"),
    department=("user_department_clean", "first"),
    employee_status=("user_status_clean", "first"),
).reset_index()
risky_users = risky_users.merge(user_reference, on="user_id_clean", how="left")

display(risky_devices.head(15))
display(risky_users.head(15))

,device_key,alert_count,critical_alert_count,high_alert_count,unresolved_alert_count,unresolved_critical_count,malware_or_ransomware_count,powershell_count,lateral_movement_count,credential_access_count,...,credential_access_points,tampering_points,usb_points,impossible_resolution_points,inactive_user_points,identity_review_points,risk_band,hostname,device_criticality,departments
0,WS-11374,7,1,2,5,1,2,0,1,1,...,2,0,1,0,21,7,Critical,WS-11374,Low,Hr
1,WS-12745,6,0,4,4,0,3,1,1,1,...,2,0,0,1,18,6,Critical,WS-12745,Medium,Cs
2,VDR-10238,7,1,2,3,1,3,3,0,0,...,0,0,0,0,21,0,Critical,VDR-10238,Medium,Legal
3,WS-10394,8,2,3,5,2,4,0,2,2,...,4,0,0,1,0,0,Critical,WS-10394,High,Human Resource
4,LPT-12990,6,2,1,5,1,2,0,0,3,...,6,0,0,0,18,0,Critical,LPT-12990,Low,Procurement Team
5,VDR-11757,6,1,2,4,1,4,1,0,0,...,0,0,0,0,18,0,Critical,VDR-11757,Low,It Support
6,LPT-11024,7,0,3,3,0,3,0,2,0,...,0,0,1,1,21,0,Critical,LPT-11024,Medium,Innovation
7,LPT-11862,7,0,2,7,0,3,2,1,1,...,2,0,0,0,21,0,Critical,LPT-11862,Medium,Cs
8,VDR-10300,6,0,4,5,0,2,1,2,0,...,0,0,1,1,18,0,Critical,VDR-10300,High,Rnd
9,VDR-10914,6,0,3,6,0,1,3,0,1,...,2,0,0,0,18,6,Critical,VDR-10914,Critical,Operations Dept


,user_id_clean,alert_count,critical_alert_count,high_alert_count,unresolved_alert_count,unresolved_critical_count,malware_or_ransomware_count,powershell_count,lateral_movement_count,credential_access_count,...,credential_access_points,tampering_points,usb_points,impossible_resolution_points,inactive_user_points,identity_review_points,risk_band,full_name,department,employee_status
0,EMP11374,8,1,2,6,1,3,0,1,1,...,2,0,1,0,24,8,Critical,Vyanjana Ahuja,Hr,Inactive
1,EMP11218,5,3,1,5,3,2,0,1,1,...,2,0,1,0,15,1,Critical,Karan Goda,Mkt,Disabled
2,EMP10099,6,2,1,6,2,2,0,1,1,...,2,0,1,0,18,5,Critical,Dev Kanda,Rnd,Disabled
3,EMP12745,6,0,4,4,0,3,1,1,1,...,2,0,0,1,18,6,Critical,Prisha Sarkar,Cs,Disabled
4,EMP10562,8,0,1,8,0,0,2,2,3,...,6,0,1,0,24,8,Critical,Manan Desai,Purch,Inactive
5,EMP10238,7,1,2,3,1,3,3,0,0,...,0,0,0,0,21,0,Critical,Sanya Narang,Legal,Inactive
6,EMP10300,7,0,4,5,0,2,1,3,0,...,0,0,1,1,21,1,Critical,Adweta Kaur,Rnd,Disabled
7,EMP11376,7,0,3,5,0,3,0,1,1,...,2,0,0,0,21,6,Critical,Matthew Dalal,R&D,Inactive
8,EMP10394,9,2,3,6,2,4,0,2,2,...,4,0,1,1,0,1,Critical,Kiaan Suri,Human Resource,NaN
9,EMP12446,7,1,2,5,1,2,1,1,1,...,2,0,0,0,21,1,Critical,Aradhana Saha,Human Resource,Disabled


## 8. Export deliverables

### Export outputs


In [43]:
# Keep a useful, auditable set of columns while retaining all raw and clean fields in the main dataset.
endpoint_output = endpoint.drop(columns=["alert_name_lower"], errors="ignore")
enriched_output = enriched.drop(columns=["alert_name_lower"], errors="ignore")

outputs = {
    "track2_endpoint_alerts_clean.csv": endpoint_output,
    "track2_endpoint_alerts_identity_enriched.csv": enriched_output,
    "track2_risky_devices.csv": risky_devices,
    "track2_risky_users.csv": risky_users,
    "track2_endpoint_alerts_quality_report.csv": quality_report,
    "track2_endpoint_alerts_join_summary.csv": enrichment_summary,
}
for filename, frame in outputs.items():
    target = (REPORT_DIR if "report" in filename or "summary" in filename else PROCESSED_DIR) / filename
    frame.to_csv(target, index=False)
    print(f"Wrote {target} ({len(frame):,} rows)")

raw_baseline.to_csv(REPORT_DIR / "track2_endpoint_alerts_raw_baseline.csv", index=False)

Wrote c:\Users\aadhi\.copilot\chats\2026-09-13\attached-file-track2-dataset-notes-txt-c-f0ad7025\data\processed\track2_endpoint_alerts_clean.csv (8,240 rows)
Wrote c:\Users\aadhi\.copilot\chats\2026-09-13\attached-file-track2-dataset-notes-txt-c-f0ad7025\data\processed\track2_endpoint_alerts_identity_enriched.csv (8,240 rows)
Wrote c:\Users\aadhi\.copilot\chats\2026-09-13\attached-file-track2-dataset-notes-txt-c-f0ad7025\data\processed\track2_risky_devices.csv (3,723 rows)
Wrote c:\Users\aadhi\.copilot\chats\2026-09-13\attached-file-track2-dataset-notes-txt-c-f0ad7025\data\processed\track2_risky_users.csv (2,799 rows)
Wrote c:\Users\aadhi\.copilot\chats\2026-09-13\attached-file-track2-dataset-notes-txt-c-f0ad7025\reports\track2_endpoint_alerts_quality_report.csv (19 rows)
Wrote c:\Users\aadhi\.copilot\chats\2026-09-13\attached-file-track2-dataset-notes-txt-c-f0ad7025\reports\track2_endpoint_alerts_join_summary.csv (3 rows)


## Deliverables produced

The notebook also displays mapping tables, quality breakdowns, join coverage, and ranked risk tables for human review.

- `data/processed/track2_endpoint_alerts_clean.csv`
- `data/processed/track2_endpoint_alerts_identity_enriched.csv`
- `data/processed/track2_risky_devices.csv`
- `data/processed/track2_risky_users.csv`
- `reports/track2_endpoint_alerts_quality_report.csv`
- `reports/track2_endpoint_alerts_join_summary.csv`
- `reports/track2_endpoint_alerts_raw_baseline.csv`
